# TildeOpen 30B 64k: Translation, Contextual QA and Summarisation on Amazon SageMaker

This notebook shows how to deploy and use **[TildeOpen 30B 64k (MT / CQA / Summarisation)](<AWS Marketplace listing URL>)** from AWS Marketplace.

TildeOpen 30B 64k is a 30.7B-parameter open European language model with a 65,536-token context window, instruction-tuned by [Tilde](https://tilde.ai) for three tasks across 30+ European languages:

1. **Machine translation**, with optional term glossaries and translation-memory examples
2. **Contextual question answering**, grounded in a supplied document, with abstention when the document lacks the answer
3. **Summarisation** of a supplied document under a free-text instruction

The notebook runs end to end in any AWS Region without additional input, provided you have subscribed to the product. It covers:

1. [Subscribe to the model package](#1.-Subscribe-to-the-model-package)
2. [Create an endpoint and perform real-time inference](#2.-Create-an-endpoint-and-perform-real-time-inference)
3. [Perform batch inference](#3.-Perform-batch-inference)
4. [Clean up](#4.-Clean-up)

**Prerequisites**

- This notebook was tested in Amazon SageMaker Studio / Notebook Instances with the `conda_python3` kernel.
- The IAM role attached to the notebook needs `AmazonSageMakerFullAccess` and permission to read and write the SageMaker default S3 bucket.
- Your account needs service quota for the instance type used below in this Region.

## 1. Subscribe to the model package

1. Open the [product listing](<AWS Marketplace listing URL>) in AWS Marketplace.
2. Choose **Continue to subscribe**, review the pricing and licence, and accept.
3. Choose **Continue to configuration**, select your Region, and note the **Product ARN**. It is also listed per Region in the cell below.

In [ ]:
import boto3, json, time, sagemaker
from sagemaker import ModelPackage

session = sagemaker.Session()
region = session.boto_region_name
role = sagemaker.get_execution_role()
bucket = session.default_bucket()
runtime = boto3.client("sagemaker-runtime")

# Model package ARN per Region. Replace the placeholders with the ARNs from the product configuration page.
# TODO(Tilde): fill in before publishing.
MODEL_PACKAGE_ARNS = {
    "us-east-1":      "arn:aws:sagemaker:us-east-1:<account>:model-package/<package-name>",
    "us-east-2":      "arn:aws:sagemaker:us-east-2:<account>:model-package/<package-name>",
    "us-west-1":      "arn:aws:sagemaker:us-west-1:<account>:model-package/<package-name>",
    "us-west-2":      "arn:aws:sagemaker:us-west-2:<account>:model-package/<package-name>",
    "ca-central-1":   "arn:aws:sagemaker:ca-central-1:<account>:model-package/<package-name>",
    "eu-central-1":   "arn:aws:sagemaker:eu-central-1:<account>:model-package/<package-name>",
    "eu-west-1":      "arn:aws:sagemaker:eu-west-1:<account>:model-package/<package-name>",
    "eu-west-2":      "arn:aws:sagemaker:eu-west-2:<account>:model-package/<package-name>",
    "eu-west-3":      "arn:aws:sagemaker:eu-west-3:<account>:model-package/<package-name>",
    "eu-north-1":     "arn:aws:sagemaker:eu-north-1:<account>:model-package/<package-name>",
    "ap-northeast-1": "arn:aws:sagemaker:ap-northeast-1:<account>:model-package/<package-name>",
    "ap-northeast-2": "arn:aws:sagemaker:ap-northeast-2:<account>:model-package/<package-name>",
    "ap-south-1":     "arn:aws:sagemaker:ap-south-1:<account>:model-package/<package-name>",
    "ap-southeast-1": "arn:aws:sagemaker:ap-southeast-1:<account>:model-package/<package-name>",
    "ap-southeast-2": "arn:aws:sagemaker:ap-southeast-2:<account>:model-package/<package-name>",
    "sa-east-1":      "arn:aws:sagemaker:sa-east-1:<account>:model-package/<package-name>",
}
if region not in MODEL_PACKAGE_ARNS:
    raise RuntimeError(f"This product is not available in {region}. Supported Regions: {sorted(MODEL_PACKAGE_ARNS)}")
model_package_arn = MODEL_PACKAGE_ARNS[region]

# Instance types. TODO(Tilde): confirm the recommended instance types before publishing.
REAL_TIME_INSTANCE_TYPE = "ml.g6e.12xlarge"
BATCH_INSTANCE_TYPE = "ml.g6e.12xlarge"

CONTENT_TYPE = "application/json"
BATCH_CONTENT_TYPE = "application/jsonlines"
MODEL_NAME = "tildeopen-30b-64k-mt-cqa-sum"

print("Region:", region)
print("Role:", role)
print("Bucket:", bucket)
print("Model package ARN:", model_package_arn)

## 2. Create an endpoint and perform real-time inference

### 2.1 Create the model and deploy the endpoint

Loading the 30B model onto the GPUs takes 10–15 minutes. The health-check timeout is raised accordingly.

In [ ]:
model = ModelPackage(
    role=role,
    model_package_arn=model_package_arn,
    sagemaker_session=session,
)

endpoint_name = f"{MODEL_NAME}-{int(time.time())}"

predictor = model.deploy(
    initial_instance_count=1,
    instance_type=REAL_TIME_INSTANCE_TYPE,
    endpoint_name=endpoint_name,
    container_startup_health_check_timeout=1800,
)
print("Endpoint in service:", endpoint_name)

### 2.2 Prompt templates

The model expects one of six fixed prompt templates. Only the variables change; the wording, order and system prompts are fixed. The helpers below build them correctly.

| Task | System prompt | User turn |
|---|---|---|
| Translation | `You are a translator.` | `{text}\n\nTranslate the given text from {src} to {trg}.` |
| Translation + glossary | `You are a translator.` | `... from {src} to {trg}, using this glossary {json}.` |
| Translation + examples | `You are a translator.` | prior user/assistant pairs in the plain format, then the request |
| Translation + glossary + examples | `You are a translator.` | examples in the plain format, request in the glossary format |
| Contextual QA | `You are answering questions.` | `{context}\n\n{question}` (question on one line) |
| Summarisation | `You are answering questions.` | `{context}\n\n{instruction}` (instruction on one line) |

In [ ]:
def translate_messages(text, src_lang, trg_lang, terms=None, examples=None):
    messages = [{"role": "system", "content": "You are a translator."}]
    for src, trg in (examples or []):
        messages.append({"role": "user", "content": f"{src}\n\nTranslate the given text from {src_lang} to {trg_lang}."})
        messages.append({"role": "assistant", "content": trg})
    instruction = f"Translate the given text from {src_lang} to {trg_lang}"
    if terms:
        instruction += f", using this glossary {json.dumps(terms, ensure_ascii=False)}"
    messages.append({"role": "user", "content": f"{text}\n\n{instruction}."})
    return messages

def qa_messages(context, question):
    assert "\n" not in question, "The question must be a single line"
    return [{"role": "system", "content": "You are answering questions."},
            {"role": "user", "content": f"{context}\n\n{question}"}]

def summary_messages(context, instruction="Summarise the given text. Focus on the main points."):
    assert "\n" not in instruction, "The instruction must be a single line"
    return [{"role": "system", "content": "You are answering questions."},
            {"role": "user", "content": f"{context}\n\n{instruction}"}]

def invoke(messages, max_tokens=256, temperature=0.0):
    body = {"messages": messages, "max_tokens": max_tokens, "temperature": temperature}
    response = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType=CONTENT_TYPE,
        Accept=CONTENT_TYPE,
        Body=json.dumps(body, ensure_ascii=False).encode("utf-8"),
    )
    result = json.loads(response["Body"].read())
    return result["choices"][0]["message"]["content"], result

### 2.3 Translation

In [ ]:
text, _ = invoke(translate_messages("I like pies.", "English", "Latvian"))
print(text)

### 2.4 Translation with a term glossary

The glossary maps a source term to a list of allowed target terms. The model uses the supplied term and inflects it as the target language requires.

In [ ]:
text, _ = invoke(translate_messages("I like pies.", "English", "Latvian",
                                   terms={"pie": ["kūka"], "like": ["patīk"]}))
print(text)

### 2.5 Translation with translation-memory examples

Previously approved translations are passed as prior turns. The model follows their style and terminology. A glossary can be combined with examples; it is attached to the final request only.

In [ ]:
examples = [("I like apples.", "Man patīk āboli."), ("I eat pies.", "Es ēdu kūkas.")]
text, _ = invoke(translate_messages("I like pies.", "English", "Latvian", examples=examples))
print("examples only:        ", text)

text, _ = invoke(translate_messages("I like pies.", "English", "Latvian",
                                   examples=examples[:1], terms={"pie": ["kūka"], "like": ["patīk"]}))
print("examples and glossary:", text)

### 2.6 Contextual question answering

The answer is grounded in the supplied context. When the context does not contain the answer, the model returns an informative refusal rather than guessing from memory.

In [ ]:
context = "Tilde was founded in Riga in 1991. It develops language technology for the Baltic and Nordic markets."

text, _ = invoke(qa_messages(context, "When was Tilde founded?"), max_tokens=128)
print("Answerable:  ", text)

text, _ = invoke(qa_messages(context, "How many employees does Tilde have?"), max_tokens=128)
print("Unanswerable:", text)

### 2.7 Summarisation

In [ ]:
with open("data/input/real-time/summarise.json", encoding="utf-8") as f:
    sample = json.load(f)

text, result = invoke(sample["messages"], max_tokens=sample["max_tokens"])
print(text)
print()
print("Token usage:", result["usage"])

### 2.8 Run all packaged real-time samples

The `data/input/real-time` folder holds one request per template. The responses are compared with the reference outputs in `data/output/real-time`.

In [ ]:
import glob, os

for path in sorted(glob.glob("data/input/real-time/*.json")):
    name = os.path.basename(path)
    with open(path, encoding="utf-8") as f:
        body = json.load(f)
    text, _ = invoke(body["messages"], max_tokens=body["max_tokens"], temperature=body.get("temperature", 0.0))
    with open(f"data/output/real-time/{name}", encoding="utf-8") as f:
        reference = json.load(f)["choices"][0]["message"]["content"]
    print(f"--- {name}")
    print("model:    ", text)
    print("reference:", reference)

### 2.9 Delete the endpoint

The endpoint is billed per instance-hour while it runs, so delete it before moving on to batch inference.

In [ ]:
predictor.delete_endpoint(delete_endpoint_config=True)
print("Deleted endpoint", endpoint_name)

## 3. Perform batch inference

Batch Transform reads a JSON Lines file with one request object per line and writes one response per line. `join_source="Input"` makes SageMaker attach each response to its request under the key `SageMakerOutput`, so no separate join is needed.

In [ ]:
prefix = f"{MODEL_NAME}/batch"
input_s3 = session.upload_data("data/input/batch/batch_input.jsonl", bucket=bucket, key_prefix=f"{prefix}/input")
output_s3 = f"s3://{bucket}/{prefix}/output"
print("Input: ", input_s3)
print("Output:", output_s3)

transformer = model.transformer(
    instance_count=1,
    instance_type=BATCH_INSTANCE_TYPE,
    strategy="SingleRecord",
    assemble_with="Line",
    accept=BATCH_CONTENT_TYPE,
    output_path=output_s3,
)

transformer.transform(
    data=input_s3,
    content_type=BATCH_CONTENT_TYPE,
    split_type="Line",
    join_source="Input",
)
transformer.wait()

In [ ]:
s3 = boto3.client("s3")
key = f"{prefix}/output/batch_input.jsonl.out"
lines = s3.get_object(Bucket=bucket, Key=key)["Body"].read().decode("utf-8").splitlines()

for line in lines:
    record = json.loads(line)
    request_text = record["messages"][-1]["content"].split("\n\n")[0]
    answer = record["SageMakerOutput"]["choices"][0]["message"]["content"]
    print("IN: ", request_text[:80])
    print("OUT:", answer[:200])
    print()

## 4. Clean up

Delete the model. Transform jobs stop by themselves once the input is processed; the endpoint was deleted in section 2.9.

In [ ]:
model.delete_model()
print("Deleted model")

### Unsubscribe

If you no longer need the product, cancel the subscription:

1. Make sure no endpoint or transform job created from this model package is still running.
2. Open [Your Software subscriptions](https://aws.amazon.com/marketplace/ai/library?productType=ml&ref_=mlmp_gitdemo_indust) in AWS Marketplace and choose **Cancel subscription** for this product.